In [1]:
import cadquery as cq
import math

In [2]:
def create_scale(base_width, base_length, tip_width, tip_length, path_length, attack_angle, num_sections, curvature_factor):
    angle_rad = math.radians(attack_angle)
    
    y_end = path_length * math.cos(angle_rad)
    z_end = path_length * math.sin(angle_rad)
    
    y_start = base_length / 2
    
    spine = (
        cq.Workplane("YZ")
        .moveTo(y_start, 0)
        .threePointArc(
            (y_start + y_end * 0.5, z_end * curvature_factor),
            (y_start + y_end, z_end)
        )
        .val()
    )
    
    def lerp(a, b, t):
        return a + (b - a) * t
    
    profiles = []
    
    Z = cq.Vector(0, 0, 1)
    Y = cq.Vector(0, 1, 0)
    X = cq.Vector(1, 0, 0)
    
    # Generate trapezoid profiles
    for i in range(num_sections):
        t = i / (num_sections - 1)
    
        point = spine.positionAt(t)
    
        t_rot = t**1.5
        normal = (Z.multiply(1 - t_rot) + Y.multiply(t_rot)).normalized()
    
        x_dir = X
        y_dir = normal.cross(x_dir).normalized()
    
        plane = cq.Plane(origin=point, xDir=x_dir, normal=normal)
    
        width = lerp(base_width, tip_width, t)
    
        bottom_len = base_length * (1 - 0.4 * t)
        top_len = lerp(base_length * 0.6, tip_length, t)
    
        pts = [
            (-bottom_len/2, -width/2),
            ( bottom_len/2, -width/2),
            ( top_len/2,    width/2),
            (-top_len/2,    width/2)
        ]
    
        wire = cq.Workplane(plane).polyline(pts).close().val()
        profiles.append(wire)
    
    scale = cq.Solid.makeLoft(profiles, ruled=False)
    return scale



In [3]:
base_width = 2.0
base_length = 5.0

tip_width = 1.0
tip_length = 1.0

path_length = 4.0        
attack_angle = 30.0

num_sections = 8
curvature_factor = 1.3

In [4]:
scale = create_scale(base_width, base_length, tip_width, tip_length, path_length, attack_angle, num_sections, curvature_factor)
display(scale)